In [ ]:
# ====================== INSTALL ======================
!pip install -q gradio transformers torch torchvision pillow opencv-python-headless

# ====================== IMPORTS ======================
import gradio as gr
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import numpy as np
import cv2

# ====================== LOAD MODEL ======================
print("Loading model...")

MODEL_NAME = "openai/clip-vit-base-patch32"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

print("Model ready on:", device)


# ====================== IMAGE PREPARATION ======================
def prepare_image(image):
    if image is None:
        return None

    if isinstance(image, np.ndarray):
        image = Image.fromarray(image.astype(np.uint8))

    return image.convert("RGB")


# ====================== METAL CLASSIFICATION ======================
def classify_material(image):
    """
    Compare a broad range of metal and non-metal descriptions.
    CLIP scores indicate visual similarity, not verified composition.
    """

    metal_prompts = [
        "a photograph of a metal object",
        "a photograph of a silver metal ring",
        "a photograph of a gold metal ring",
        "a photograph of a steel ring",
        "a photograph of a metal coin",
        "a photograph of a metal key",
        "a photograph of a metal spoon",
        "a photograph of a steel bolt or nut",
        "a photograph of a metal cube",
        "a photograph of a metal block",
        "a photograph of a metal machine component",
        "a photograph of an iron casting",
        "a photograph of an aluminum casting",
        "a photograph of a metal casting component",
        "a close-up photograph of a metallic surface",
        "a photograph of a copper object",
        "a photograph of a stainless steel object",
        "a photograph of a metal tool"
    ]

    nonmetal_prompts = [
        "a photograph of a wooden object",
        "a photograph of a plastic object",
        "a photograph of a rubber object",
        "a photograph of a cardboard object",
        "a photograph of a paper object",
        "a photograph of a fabric object",
        "a photograph of a fruit",
        "a photograph of a vegetable",
        "a photograph of a human",
        "a photograph of an animal",
        "a photograph of a ceramic object",
        "a photograph of a stone",
        "a photograph of a wooden toy cube",
        "a photograph of a plastic toy",
        "a photograph of a book",
        "a photograph of a flower"
    ]

    prompts = metal_prompts + nonmetal_prompts

    inputs = processor(
        text=prompts,
        images=image,
        return_tensors="pt",
        padding=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

        # Softmax over all descriptions for relative comparison.
        probabilities = outputs.logits_per_image.softmax(dim=1)[0]

    n_metal = len(metal_prompts)

    metal_score = probabilities[:n_metal].sum().item()
    nonmetal_score = probabilities[n_metal:].sum().item()

    # Relative ranking only; not a calibrated material probability.
    is_metal = metal_score > nonmetal_score

    return is_metal, metal_score, nonmetal_score


# ====================== DEFECT INSPECTION ======================
def inspect_condition(image):
    prompts = [
        # Apparently complete
        "a complete intact metal object with no visible damage",
        "an undamaged metal component with a smooth intact surface",
        "a complete metal part without visible cracks or broken sections",
        "a metal object in good visible condition",

        # Visible damage
        "a visibly cracked metal object",
        "a broken metal object with a missing section",
        "a metal object with a visibly damaged surface",
        "a metal component with obvious holes, fractures or broken edges"
    ]

    inputs = processor(
        text=prompts,
        images=image,
        return_tensors="pt",
        padding=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = outputs.logits_per_image.softmax(dim=1)[0]

    intact_score = probabilities[:4].mean().item()
    defect_score = probabilities[4:].mean().item()

    # This is a prototype screening rule, not a validated defect detector.
    if defect_score > intact_score + 0.10:
        status = "DEFECTED"
        decision = "Visible defect suspected"
        color = (0, 0, 220)
    else:
        status = "CLEAR"
        decision = "No obvious defect predicted"
        color = (0, 180, 0)

    message = (
        f"Material: METAL MATCHED\n"
        f"Status: {status}\n"
        f"Decision: {decision}\n\n"
        f"Relative intact score: {intact_score * 100:.1f}%\n"
        f"Relative defect score: {defect_score * 100:.1f}%\n\n"
        "Note: These are relative model scores, not calibrated "
        "probabilities. Internal cracks and porosity cannot be "
        "reliably ruled out from an ordinary photograph."
    )

    return status, color, message


# ====================== MAIN DETECTION FUNCTION ======================
def detect_defect(image):

    if image is None:
        return None, "Please upload or capture an image."

    pil_image = prepare_image(image)

    # STEP 1: MATERIAL SCREENING
    is_metal, metal_score, nonmetal_score = classify_material(
        pil_image
    )

    if not is_metal:
        return np.array(pil_image), (
            "NO METAL FOUND\n\n"
            "The image matched non-metal descriptions more strongly.\n"
            "Try a clear, well-lit image with the object filling "
            "most of the frame.\n\n"
            "Important: This is a visual prediction, not proof "
            "that the object contains no metal."
        )

    # STEP 2: INSPECT VISIBLE CONDITION
    status, color, message = inspect_condition(pil_image)

    # STEP 3: DRAW STATUS
    img_bgr = cv2.cvtColor(
        np.array(pil_image),
        cv2.COLOR_RGB2BGR
    )

    height, width = img_bgr.shape[:2]

    if width > 1 and height > 1:
        box_width = min(width - 1, 520)
        box_height = min(height - 1, 70)

        cv2.rectangle(
            img_bgr,
            (0, 0),
            (box_width, box_height),
            (0, 0, 0),
            -1
        )

        cv2.putText(
            img_bgr,
            f"METAL | {status}",
            (10, min(box_height - 5, 45)),
            cv2.FONT_HERSHEY_SIMPLEX,
            min(0.9, max(0.4, width / 600)),
            color,
            2
        )

    result = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    return result, message


# ====================== GRADIO INTERFACE ======================
with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="blue"),
    title="Metal Detection and Defect Inspection"
) as demo:

    gr.Markdown("""
    # Metal Detection and Defect Inspection
    Upload a photo or use your camera.

    **Workflow**
    1. Compare metal and non-metal image descriptions.
    2. If metal descriptions score higher, inspect visible condition.
    3. Display METAL + CLEAR or METAL + DEFECTED.
    4. If non-metal descriptions score higher, display NO METAL FOUND.

    This is an experimental AI prototype, not a certified
    industrial inspection system.
    """)

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(
                type="numpy",
                sources=["upload", "webcam"],
                label="Upload or Capture Object",
                height=360
            )

            run_button = gr.Button(
                "Run Inspection",
                variant="primary",
                size="lg"
            )

        with gr.Column():
            output_image = gr.Image(
                label="Inspection Result",
                height=360
            )

            output_text = gr.Textbox(
                label="Detection Report",
                lines=12,
                interactive=False
            )

    run_button.click(
        fn=detect_defect,
        inputs=input_image,
        outputs=[output_image, output_text]
    )

    gr.Markdown("""
    ---
    **Stack:** Python, PyTorch, CLIP, OpenCV and Gradio.

    For dependable material detection, train and validate a
    metal/non-metal classifier using labeled photographs.
    For industrial defect detection, use a separate model trained
    on real defect-free and defective samples.
    """)

demo.launch(share=True)

Loading model...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

Model ready on: cpu


/tmp/ipykernel_4303/315475276.py:229: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://67e4f8a94e92031cd5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
